In [1]:
keywords = {
    "Anggaran": [
        "dana",
        "anggaran",
        "apbn",
        "biaya",
    ],

    "Politik":[
        "prabowo",
        "kampanye",
        "politik",

    ],

    "Kualitan Pangan":[
        'bergizi',
        'belatung',
        'keracunan',
        'makanan',
    ],
    "Distribusi":[
        'distribusi'
        'penyaluran'
        'pengiriman'
        'daerah'
        'sampai'
    ],

    "Ekonomi":[
        'ekonomi',
        'umkm',
        'lapangan',
        'lokal',
        'kerja',
        'masyarakat',
        'banyak',
    ],

    "Tata Kelola":[
        'aturan'
        'regulasi'
        'pengawasan'
        'evaluasi'
        'pelaksanaan'
        'mekanisme'
        'koordinasi'
        'kementerian'
        'bgn'
        'prosedur'
    ],

    "Sasaran Penerima":[
        'siswa'
        'anak sekolah'
        'pelajar'
        'penerima'
        'target'
        'anak sd'
        'murid'
    ],

    "lainnya":[
        'makanan'
    ]
}

In [2]:
#import libary yang dibutuhkan
from collections import Counter
import re
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, ArrayDictionary, StopWordRemover
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\zyog1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\zyog1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [7]:
# buat nyari kata yang sering muncul sesuai label yg diminta



text = " ".join(
    train[train['label'] == 'lainnya']['full_text']
)
words = text.lower().split()
counter = Counter(words)
print(counter.most_common(90))


[]


In [8]:
#buat baca datanya


train = pd.read_excel("excel/case_1_labeled_data.xlsx")
test = pd.read_excel("excel/case_1_text_to_predict.xlsx")
# train['clean_text'] = train['text'].apply(clean_text)
# test['clean_text'] = test['text'].apply(clean_text)

train.head()


,full_text,label
0,@ARSIPAJA Pret. Di sekolah gw dapet MBG tetep ...,Sasaran Penerima
1,MBG bentuk penggarongan duit negara secara TSM...,Politik
2,@inzhapt_76 @ARSIPAJA Pasal 34 ayat (1) Undang...,Sasaran Penerima
3,Makan Bergizi Gratis bikin masyarakat ngerasa ...,Sasaran Penerima
4,"@OniSuryaman Presiden ngotot, paling sebel kal...",Politik


In [13]:
# 1. PROSES CASE FOLDING

def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'@\w+', ' ', text)                          # remove username
        text = re.sub(r'(wk){2,}k?', '', text, flags=re.IGNORECASE)  # wkwk, wkwkk dst
        text = re.sub(r'(ha){2,}', '', text, flags=re.IGNORECASE)  # haha, hahaha dst
        text = re.sub(r'(hi){2,}', '', text, flags=re.IGNORECASE)  # hihi, hihihi dst
        text = re.sub(r'(he){2,}', '', text, flags=re.IGNORECASE)  # hehe, hehehe dst
        text = re.sub(r'\s+', ' ', text).strip()                   # rapikan spasi
        text = text.lower()                                         # case folding
    return text

test['clean_text'] = test['full_text'].apply(clean_text)
test['label'] = train['label']

print("Setelah Case Folding + Remove Username:")
display(test[['full_text', 'label', 'clean_text']].head())

Setelah Case Folding + Remove Username:


,full_text,label,clean_text
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg di sekolah kota saya belum ada yg dpt mala...
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,didaerah ku pun yg dapat mbg baru beberapa sek...
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,"tai tau enggak ,gak semua sekolah dapat mbg da..."
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,"gencar cegah stunting, mbg telah capai 49% sas..."
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,140 siswa kupang keracunan program mbg


In [17]:
def remove_special_characters(text: str) -> str:
    #remove karakter spesial
    processed_text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return processed_text

test['clean_text'] = test['clean_text'].apply(remove_special_characters)

print("Setelah Remove Special Characters:")
test['label'] = train['label']
display(test[['full_text', 'label', 'clean_text']].head())

Setelah Remove Special Characters:


,full_text,label,clean_text
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg di sekolah kota saya belum ada yg dpt malahan
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,didaerah ku pun yg dapat mbg baru beberapa sek...
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,tai tau enggak gak semua sekolah dapat mbg dan...
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,gencar cegah stunting mbg telah capai 49 sasar...
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,140 siswa kupang keracunan program mbg


In [20]:
def removedigitangka(text: str) -> str:
    # menghilangkan digit angka
    text = re.sub(r'\d+', '', text)
    return text

test['clean_text'] = test['clean_text'].apply(removedigitangka)

print("Setelah Remove digit:")
test['label'] = train['label']
display(test[['full_text', 'label', 'clean_text']].head())

Setelah Remove digit:


,full_text,label,clean_text
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg di sekolah kota saya belum ada yg dpt malahan
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,didaerah ku pun yg dapat mbg baru beberapa sek...
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,tai tau enggak gak semua sekolah dapat mbg dan...
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,gencar cegah stunting mbg telah capai sasaran...
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,siswa kupang keracunan program mbg


In [23]:
def removeurl(text: str) -> str:
    # menghilangkan url
    text = re.sub(r'https?:\/\/\S+','',text)
    return text

test['clean_text'] = test['clean_text'].apply(removeurl)

print("Setelah Remove url:")
test['label'] = train['label']
display(test[['full_text', 'label', 'clean_text']].head())

Setelah Remove url:


,full_text,label,clean_text
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg di sekolah kota saya belum ada yg dpt malahan
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,didaerah ku pun yg dapat mbg baru beberapa sek...
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,tai tau enggak gak semua sekolah dapat mbg dan...
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,gencar cegah stunting mbg telah capai sasaran...
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,siswa kupang keracunan program mbg


In [24]:
# Fungsi untuk menghapus HTML
def remove_html(text: str) ->str:
    if text is not None and isinstance(test, str):
        html = re.compile(r'<.*?>')
        return html.sub(r'', text)
    else:
        return text
    
test['clean_text'] = test['clean_text'].apply(remove_html)

print("Setelah Remove html:")
test['label'] = train['label']
display(test[['full_text', 'label', 'clean_text']].head())

Setelah Remove html:


,full_text,label,clean_text
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg di sekolah kota saya belum ada yg dpt malahan
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,didaerah ku pun yg dapat mbg baru beberapa sek...
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,tai tau enggak gak semua sekolah dapat mbg dan...
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,gencar cegah stunting mbg telah capai sasaran...
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,siswa kupang keracunan program mbg


In [25]:
def remove_emoji(text: str) -> str:
    if text is not None and isinstance(text, str):
        emoji_pattern = re.compile("["
            u"\U0001F600-\U0001F64F"  # emoticons
            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # transport & map symbols
            u"\U0001F700-\U0001F77F"  # alchemical symbols
            u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
            u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
            u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
            u"\U0001FA00-\U0001FA6F"  # Chess Symbols
            u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
            u"\U0001F004-\U0001F0CF"  # Additional emoticons
            u"\U0001F1E0-\U0001F1FF"  # flags
                               "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', text)
    else:
        return text
    
test['clean_text'] = test['clean_text'].apply(remove_html)

print("Setelah Remove emoji:")
test['label'] = train['label']
display(test[['full_text', 'label', 'clean_text']].head())

Setelah Remove emoji:


,full_text,label,clean_text
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg di sekolah kota saya belum ada yg dpt malahan
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,didaerah ku pun yg dapat mbg baru beberapa sek...
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,tai tau enggak gak semua sekolah dapat mbg dan...
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,gencar cegah stunting mbg telah capai sasaran...
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,siswa kupang keracunan program mbg


In [26]:
def stop_stem(text):
    # Stopword removal
    with open('Kamus/kamus.txt') as kamus:
        word = kamus.readlines()
        list_stopword = [line.replace('\n', '') for line in word]
    dictionary = ArrayDictionary(list_stopword)
    stopword = StopWordRemover(dictionary)
    text = stopword.remove(text)

    # Stemming
    factory_stemmer = StemmerFactory()
    stemmer = factory_stemmer.create_stemmer()
    text = stemmer.stem(text)

    return text

test['clean_text'] = test['clean_text'].apply(stop_stem)

print("Setelah Stopword Removal + Stemming:")
test['label'] = train['label']
display(test[['full_text', 'label', 'clean_text']].head())

Setelah Stopword Removal + Stemming:


,full_text,label,clean_text
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg sekolah kota saya belum ada dpt malah
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,daerah ku dapat mbg baru beberapa seko
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,tai tau enggak gak semua sekolah dapat mbg mas...
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,gencar cegah stunting mbg telah capai sasar ib...
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,siswa kupang racun program mbg


In [27]:
def word_tokenize_wrapper(text):
    return word_tokenize(text)

test['tweet_tokens'] = test['clean_text'].apply(word_tokenize_wrapper)

test['label'] = train['label']
display(test[['full_text', 'label', 'clean_text', 'tweet_tokens']].head())

,full_text,label,clean_text,tweet_tokens
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg sekolah kota saya belum ada dpt malah,"[mbg, sekolah, kota, saya, belum, ada, dpt, ma..."
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,daerah ku dapat mbg baru beberapa seko,"[daerah, ku, dapat, mbg, baru, beberapa, seko]"
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,tai tau enggak gak semua sekolah dapat mbg mas...,"[tai, tau, enggak, gak, semua, sekolah, dapat,..."
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,gencar cegah stunting mbg telah capai sasar ib...,"[gencar, cegah, stunting, mbg, telah, capai, s..."
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,siswa kupang racun program mbg,"[siswa, kupang, racun, program, mbg]"


In [29]:
kamus_normalisasi = pd.read_csv("Kamus/slang.csv")

kata_normalisasi_dict = {}
for index, row in kamus_normalisasi.iterrows():
    if row['slang'] not in kata_normalisasi_dict:
        kata_normalisasi_dict[row['slang']] = row['formal']

def normalisasi_kata(document):
    return [kata_normalisasi_dict[term] if term in kata_normalisasi_dict else term for term in document]

test['tweet_tokens'] = test['clean_text'].apply(word_tokenize)
test['normalisasi'] = test['tweet_tokens'].apply(normalisasi_kata)
test['label'] = train['label']
print("Setelah Normalisasi:")
display(test[['full_text','label', 'clean_text', 'normalisasi']].head())

Setelah Normalisasi:


,full_text,label,clean_text,normalisasi
0,@yeahhhmaybe mbg di sekolah kota saya belum ad...,Sasaran Penerima,mbg sekolah kota saya belum ada dpt malah,"[mbg, sekolah, kota, aku, belum, ada, dapat, m..."
1,@thefineshytguy WKWKKdidaerah ku pun yg dapat ...,Politik,daerah ku dapat mbg baru beberapa seko,"[daerah, ku, dapat, mbg, baru, beberapa, dari]"
2,"@ARSIPAJA Tai tau enggak ,gak semua sekolah da...",Sasaran Penerima,tai tau enggak gak semua sekolah dapat mbg mas...,"[tahi, tau, tidak, tidak, semua, sekolah, dapa..."
3,"Gencar Cegah Stunting, MBG Telah Capai 49% Sas...",Sasaran Penerima,gencar cegah stunting mbg telah capai sasar ib...,"[gencar, cegah, stunting, mbg, telah, capai, s..."
4,140 siswa kupang Keracunan program MBG @prabowo,Politik,siswa kupang racun program mbg,"[siswa, kupang, racun, program, mbg]"


In [ ]:
import sys

!{sys.executable} -m pip install PySastrawi
!{sys.executable} -m pip install openpyxl


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, ArrayDictionary, StopWordRemover
print("Import berhasil!")

Import berhasil!
